# VLM Modality Benchmark — 8 Models, $0 Cost

All models are **100% free and open-source**, running on Colab T4 or Kaggle P100.

## Model Registry

| # | Model | Params | VRAM (4-bit) | Est. Time |
|---|-------|--------|-------------|----------|
| 1 | Qwen2-VL-2B | 2B | ~3GB | ~1.5h |
| 2 | LLaVA-1.6-7B | 7B | ~5GB | ~2.5h |
| 3 | Qwen2.5-VL-7B | 7B | ~6GB | ~2.5h |
| 4 | InternVL2-8B | 8B | ~6GB | ~3h |
| 5 | LLaVA-OneVision-7B | 7B | ~5GB | ~2.5h |
| 6 | Phi-3.5-Vision | 4.2B | ~4GB | ~2h |
| 7 | MiniCPM-V-2.6 | 8B | ~6GB | ~3h |
| 8 | Idefics3-8B | 8B | ~6GB | ~3h |

**Strategy:** Run 2-3 models per Colab session. Save results to Google Drive.
Resume in the next session by changing `MODELS_TO_RUN`.

---

In [ ]:
# ── Install dependencies ──
!pip install torch transformers datasets scipy pyyaml tqdm pillow bitsandbytes --quiet
!pip install transformers>=4.45.0 --quiet
# For InternVL2
!pip install torchvision --quiet

In [ ]:
# ── Clone repo & mount Drive ──
!git clone https://github.com/Ro-netizen004/vlm-modality-research.git /content/vlm-modality-research 2>/dev/null || echo 'Already cloned'

import sys
sys.path.insert(0, '/content/vlm-modality-research')

from google.colab import drive
drive.mount('/content/drive')

# Results persist across sessions via Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/vlm_research_results'
!mkdir -p {DRIVE_OUTPUT}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIGURATION — CHANGE THIS EACH SESSION
# ══════════════════════════════════════════════════════════════════════════

# Which models to run THIS session (by index, 0-indexed).
# Session 1: models 0,1,2  (~6.5h)
# Session 2: models 3,4,5  (~7.5h)
# Session 3: models 6,7    (~6h)

MODELS_TO_RUN = [0, 1, 2]  # ◀── CHANGE THIS EACH SESSION

NUM_PROBLEMS = None  # None = full 1319 test set, or 100 for quick test

# Full model registry
ALL_MODELS = [
    # 0
    {'name': 'Qwen/Qwen2-VL-2B-Instruct', 'type': 'qwen',
     'max_new_tokens': 256, 'torch_dtype': 'bfloat16', 'quantize': None},
    # 1
    {'name': 'llava-hf/llava-v1.6-mistral-7b-hf', 'type': 'llava',
     'max_new_tokens': 512, 'torch_dtype': 'float16', 'quantize': '4bit'},
    # 2
    {'name': 'Qwen/Qwen2.5-VL-7B-Instruct', 'type': 'qwen',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
    # 3
    {'name': 'OpenGVLab/InternVL2-8B', 'type': 'internvl',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
    # 4
    {'name': 'llava-hf/llava-onevision-qwen2-7b-ov-hf', 'type': 'llava_onevision',
     'max_new_tokens': 512, 'torch_dtype': 'float16', 'quantize': '4bit'},
    # 5
    {'name': 'microsoft/Phi-3.5-vision-instruct', 'type': 'phi',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
    # 6
    {'name': 'openbmb/MiniCPM-V-2_6', 'type': 'minicpm',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
    # 7
    {'name': 'HuggingFaceM4/Idefics3-8B-Llama3', 'type': 'idefics',
     'max_new_tokens': 512, 'torch_dtype': 'bfloat16', 'quantize': '4bit'},
]

SELECTED = [ALL_MODELS[i] for i in MODELS_TO_RUN]
print(f'This session: {[m["name"].split("/")[-1] for m in SELECTED]}')

In [ ]:
# ── Load dataset & render images ──
import os, torch
from datasets import load_dataset
from src.rendering import render_all_images

torch.manual_seed(42)

full_dataset = load_dataset('openai/gsm8k', 'main', split='test')
if NUM_PROBLEMS:
    full_dataset = full_dataset.select(range(NUM_PROBLEMS))

questions = list(full_dataset['question'])
references = list(full_dataset['answer'])
N = len(questions)
print(f'Problems: {N}')

IMAGE_DIR = os.path.join(DRIVE_OUTPUT, 'rendered_images')
render_all_images(questions, IMAGE_DIR)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  MAIN BENCHMARK LOOP
# ══════════════════════════════════════════════════════════════════════════

import time, json
import pandas as pd
from collections import Counter
from tqdm import tqdm

from src.models import VLMModel
from src.evaluation import (
    answers_match, classify_error, compute_accuracy,
    compute_all_statistics, format_statistics_report,
    extract_numeric_answer,
)
from src.rendering import load_image
from src.visualization import plot_error_breakdown, plot_mismatch_dominance

for mc in SELECTED:
    short = mc['name'].split('/')[-1]
    model_dir = os.path.join(DRIVE_OUTPUT, short)
    
    # Skip if already completed
    if os.path.exists(os.path.join(model_dir, 'statistics.json')):
        print(f'\n>>> SKIPPING {short} — results already exist in Drive <<<')
        continue
    
    os.makedirs(model_dir, exist_ok=True)
    print(f"\n{'='*70}")
    print(f'  MODEL: {mc["name"]}')
    print(f"{'='*70}\n")
    
    vlm = VLMModel(
        model_name=mc['name'], model_type=mc['type'],
        max_new_tokens=mc.get('max_new_tokens', 256),
        torch_dtype=mc.get('torch_dtype', 'bfloat16'),
        quantize=mc.get('quantize'),
    )
    vlm.load()
    t0 = time.time()
    
    # ── Condition 1: Text-Only ──────────────────────────────────────────
    print('Condition 1: Text-Only')
    text_preds, text_correct, text_errors = [], [], []
    for q, ref in tqdm(zip(questions, references), total=N, desc='Text'):
        try:
            pred = vlm.generate_text_only(q)
        except Exception as e:
            pred = f'ERROR: {e}'
        text_preds.append(pred)
        text_correct.append(answers_match(pred, ref))
        text_errors.append(classify_error(pred, ref))
    print(f'  Accuracy: {compute_accuracy(text_correct):.3f}')
    
    # ── Condition 2: Rendered Image ────────────────────────────────────
    print('\nCondition 2: Rendered Image')
    img_preds, img_correct, img_errors = [], [], []
    for i in tqdm(range(N), desc='Image'):
        try:
            img = load_image(i, IMAGE_DIR)
            pred = vlm.generate_with_image(img)
        except Exception as e:
            pred = f'ERROR: {e}'
        img_preds.append(pred)
        img_correct.append(answers_match(pred, references[i]))
        img_errors.append(classify_error(pred, references[i]))
    print(f'  Accuracy: {compute_accuracy(img_correct):.3f}')
    
    # ── Condition 3: Mismatch ──────────────────────────────────────────
    print('\nCondition 3: Mismatch')
    mm_preds, mm_follows = [], []
    for i in tqdm(range(N), desc='Mismatch'):
        txt_idx = (i + 1) % N
        prompt = (f'Solve the following math problem step by step. '
                  f"End with '#### <answer>'.\n\nProblem: {questions[txt_idx]}")
        try:
            img = load_image(i, IMAGE_DIR)
            pred = vlm.generate_with_image(img, text_prompt=prompt)
        except Exception as e:
            pred = f'ERROR: {e}'
        
        pred_val = extract_numeric_answer(pred)
        img_val  = extract_numeric_answer(references[i])
        txt_val  = extract_numeric_answer(references[txt_idx])
        
        if pred_val is None:
            follows = 'invalid'
        elif img_val is None or txt_val is None:
            follows = 'invalid'
        elif img_val == txt_val:
            follows = 'ambiguous'
        # TODO: Phase 3 — round() works for GSM8K integers but breaks for
        # SVAMP decimals and AQuA-RAT letter answers. Add per-dataset matching.
        elif round(pred_val) == round(img_val):
            follows = 'image'
        elif round(pred_val) == round(txt_val):
            follows = 'text'
        else:
            follows = 'neither'
        
        mm_preds.append(pred)
        mm_follows.append(follows)
    
    elapsed = time.time() - t0
    print(f'\nDominance: {Counter(mm_follows)}')
    print(f'Total time: {elapsed/60:.1f} min')
    
    # ── Statistics ─────────────────────────────────────────────────────
    stats = compute_all_statistics(text_correct, img_correct, mm_follows)
    stats['elapsed_minutes'] = round(elapsed / 60, 1)
    print(format_statistics_report(stats))
    
    # ── Save to Drive ──────────────────────────────────────────────────
    pd.DataFrame({
        'problem_id': range(N), 'question': questions, 'reference': references,
        'pred_text': text_preds, 'correct_text': text_correct, 'error_text': text_errors,
        'pred_rendered': img_preds, 'correct_rendered': img_correct, 'error_rendered': img_errors,
        'pred_mismatch': mm_preds, 'mismatch_follows': mm_follows,
    }).to_csv(os.path.join(model_dir, 'gsm8k_results.csv'), index=False)
    
    with open(os.path.join(model_dir, 'statistics.json'), 'w') as f:
        serializable = {k: (list(v) if isinstance(v, tuple) else v) for k, v in stats.items()}
        json.dump(serializable, f, indent=2)
    
    with open(os.path.join(model_dir, 'statistics_report.txt'), 'w') as f:
        f.write(format_statistics_report(stats))
    
    plot_error_breakdown({'Text-Only': text_errors, 'Rendered Image': img_errors},
                         mc['name'], model_dir)
    plot_mismatch_dominance(Counter(mm_follows), mc['name'], model_dir)
    
    print(f'\nResults saved to: {model_dir}')
    vlm.unload()

print('\n\nSession complete!')

## Cross-Model Comparison (run after ALL models are done)

Once all 8 models have results in Drive, run this cell to generate comparison plots.

In [ ]:
# ── Aggregate results from all completed models ──
import json, os
import pandas as pd
from src.visualization import plot_accuracy_comparison, plot_statistical_summary

all_stats = {}
for entry in os.listdir(DRIVE_OUTPUT):
    stats_path = os.path.join(DRIVE_OUTPUT, entry, 'statistics.json')
    if os.path.isfile(stats_path):
        with open(stats_path) as f:
            s = json.load(f)
            # Convert lists back to tuples for CI fields
            for k in ['text_ci_95', 'img_ci_95', 'text_boot_ci_95', 'img_boot_ci_95']:
                if k in s and isinstance(s[k], list):
                    s[k] = tuple(s[k])
            all_stats[entry] = s

print(f'Found results for {len(all_stats)} models:')
for m in sorted(all_stats.keys()):
    s = all_stats[m]
    sig = '*' if s['mcnemar_p'] < 0.05 else ''
    print(f"  {m:40s}  Text={s['acc_text']:.3f}  Image={s['acc_img']:.3f}  "
          f"Drop={s['acc_drop']:+.3f}  p={s['mcnemar_p']:.4f}{sig}")

In [ ]:
# ── Generate comparison plots ──
if len(all_stats) >= 2:
    plot_accuracy_comparison(all_stats, DRIVE_OUTPUT)
    plot_statistical_summary(all_stats, DRIVE_OUTPUT)
    print('Comparison plots saved!')

# ── Summary table ──
rows = []
for m in sorted(all_stats.keys()):
    s = all_stats[m]
    rows.append({
        'Model': m,
        'N': s['n'],
        'Text Acc': f"{s['acc_text']:.3f}",
        'Text 95% CI': f"[{s['text_ci_95'][0]:.3f}, {s['text_ci_95'][1]:.3f}]",
        'Image Acc': f"{s['acc_img']:.3f}",
        'Image 95% CI': f"[{s['img_ci_95'][0]:.3f}, {s['img_ci_95'][1]:.3f}]",
        'Drop (pp)': f"{s['acc_drop']*100:+.1f}",
        "Cohen's h": f"{s['cohens_h']:.3f}",
        'McNemar p': f"{s['mcnemar_p']:.2e}",
        'Significant': '***' if s['mcnemar_p']<0.001 else '**' if s['mcnemar_p']<0.01 else '*' if s['mcnemar_p']<0.05 else 'ns',
        'Decidable': s['mismatch_counts'].get('image',0) + s['mismatch_counts'].get('text',0),
        'Neither': s['mismatch_counts'].get('neither',0),
        'Ambiguous': s['mismatch_counts'].get('ambiguous',0),
        'Text Pref %': f"{s['mismatch_counts'].get('text',0) / max(s['mismatch_counts'].get('image',0)+s['mismatch_counts'].get('text',0),1)*100:.1f}%",
        'Image Pref %': f"{s['mismatch_counts'].get('image',0) / max(s['mismatch_counts'].get('image',0)+s['mismatch_counts'].get('text',0),1)*100:.1f}%",
    })

summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(os.path.join(DRIVE_OUTPUT, 'cross_model_summary.csv'), index=False)
print(f"\nSummary saved to {DRIVE_OUTPUT}/cross_model_summary.csv")

In [ ]:
# ── Download all results ──
!cd {DRIVE_OUTPUT} && zip -r /content/vlm_all_results.zip . -x '*/rendered_images/*'
from google.colab import files
files.download('/content/vlm_all_results.zip')